<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/pemrosesantekspraktikpert7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import polars as pl
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [2]:
CSV_PATH = "/content/hasil_preprocessingdatasample.csv"
TEXT_COL = "teks"
MODEL_NAME = "mdhugol/indonesia-bert-sentiment-classification"
MODE = "binary" # "binary" (positif/negatif) atau "three" (+ netral)
BATCH_SIZE = 32
MAX_LEN = 128
ID2LABEL = {0: "positif", 1: "netral", 2: "negatif"}


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()

config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

In [7]:
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

@torch.no_grad()
def prediksi(teks_list):
    labels, skor = [], []
    for i in range(0, len(teks_list), BATCH_SIZE):
        batch = [str(t) if t is not None else "" for t in teks_list[i:i + BATCH_SIZE]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
        logits = model(**enc).logits.cpu().numpy()
        if MODE == "binary":
            logits = logits[:, [0, 2]]  # hanya positif & negatif
            ids = np.array([0, 2])
        else:
            ids = np.array([0, 1, 2])
        prob = softmax(logits)
        pilih = prob.argmax(axis=1)
        labels += [ID2LABEL[ids[j]] for j in pilih]
        skor += prob.max(axis=1).round(4).tolist()
    return labels, skor

In [16]:
df = pl.read_csv(CSV_PATH)
# Deteksi otomatis kolom teks jika 'teks' tidak ditemukan
text_column = TEXT_COL if TEXT_COL in df.columns else [col for col in df.columns if "text" in col][0]

label, skor = prediksi(df[text_column].to_list())
df_label = df.with_columns(
    pl.Series("label", label),
    pl.Series("skor", skor), # confidence, berguna untuk cek manual
)

In [15]:
# Menampilkan dataframe df_label yang berisi hasil prediksi dan kolom tokens_lemmatized
df = pl.read_csv(CSV_PATH)
df

type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,lang,bookmarkCount,isReply,inReplyToId,conversationId,inReplyToUserId,inReplyToUsername,isPinned,author,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized
str,f64,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,bool,f64,f64,f64,str,bool,str,str,str,str,str,str,str,str,str,bool,i64,bool,str,str,str,str,str
"""tweet""",2.1153e18,"""https://x.com/arifin34533/stat…","""https://twitter.com/arifin3453…","""MATI‼️ https://t.co/Tn9mf4vwE8""",null,0,0,0,0,4,"""Wed Oct 28 04:56:49 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""","""{'binding_values': [{'key': 'p…","""{}""","""{'urls': [{'display_url': 'tem…",null,null,null,null,false,0,null,"""MATI‼""","""mati!!""","""mati""","""mati""","""mati"""
"""tweet""",2.1153e18,"""https://x.com/trynthi_/status/…","""https://twitter.com/trynthi_/s…","""Lengkap banget perlengkapannya…",null,0,0,0,0,15,"""Wed Oct 28 04:56:06 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,null,null,null,false,0,null,"""Lengkap banget perlengkapannya…","""lengkap sekali perlengkapannya…","""lengkap sekali perlengkapannya…","""lengkap perlengkapannya ngadep…","""lengkap perlengkapan ngadepin …"
"""tweet""",2.1153e18,"""https://x.com/tempodotco/statu…","""https://twitter.com/tempodotco…","""Baca: https://t.co/TVLqbdOxCs""",null,0,0,1,0,60,"""Wed Oct 28 04:56:05 +0000 2026""","""in""",0,true,2.1153e18,2.1153e18,1.8129942e7,"""tempodotco""",false,"""{'type': 'user', 'userName': '…","""{}""","""{'binding_values': [{'key': 'p…","""{}""","""{'urls': [{'display_url': 'tem…","""{'rest_id': '18129942', 'resul…",null,null,null,false,0,null,"""Baca:""","""baca:""","""baca""","""baca""","""baca"""
"""tweet""",2.1153e18,"""https://x.com/twit_boongan/sta…","""https://twitter.com/twit_boong…","""@PocongGaib @prabowo @DivHumas…",null,0,0,0,0,4,"""Wed Oct 28 04:55:21 +0000 2026""","""in""",0,true,2.1153e18,2.1153e18,8.7500011e7,"""PocongGaib""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'user_mentions': [{'id_str': …","""{'rest_id': '87500011', 'resul…",null,null,null,false,0,null,"""dari segitu banyaknya tni dan …","""dari segitu banyaknya tni dan …","""dari segitu banyaknya tni dan …","""segitu banyaknya tni polisi kn…","""segitu banyak tni polisi knp t…"
"""tweet""",2.1153e18,"""https://x.com/candysvga/status…","""https://twitter.com/candysvga/…","""14ribu orang ini kalau diturun…",null,0,0,0,0,4,"""Wed Oct 28 04:54:45 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,"""{'rest_id': '21152711581532737…","""{'type': 'tweet', 'id': '21152…",null,false,0,true,"""14ribu orang ini kalau diturun…","""14ribu orang ini kalau diturun…","""14ribu orang ini kalau diturun…","""14ribu orang diturunkan padami…","""14ribu orang turun padamin bak…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tweet""",2.1146e18,"""https://x.com/officialinews_/s…","""https://twitter.com/officialin…","""Kebakaran hutan dan lahan (kar…",null,2,1,9,0,844,"""Mon Oct 26 02:56:06 +0000 2026""","""in""",0,false,null,2.1146e18,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{'hashtags': [{'indices': [728…",null,null,null,null,false,0,null,"""Kebakaran hutan dan lahan (kar…","""kebakaran hutan dan lahan (kar…","""kebakaran hutan dan lahan karh…","""kebakaran hutan lahan karhutla…","""bakar hutan lahan karhutla kaw…"
"""tweet""",2.1146e18,"""https://x.com/InfoHumasBMKG/st…","""https://twitter.com/InfoHumasB…","""Halo #SobatBMKG! 👋🏻 Pada 25 O…",null,1,0,1,0,1809,

In [19]:
# Gunakan dataframe hasil prediksi yang sudah ada
df = df_label
df

type,id,url,twitterUrl,text,source,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,lang,bookmarkCount,isReply,inReplyToId,conversationId,inReplyToUserId,inReplyToUsername,isPinned,author,extendedEntities,card,place,entities,reply_to_user_results,quoted_tweet_results,quoted_tweet,retweeted_tweet,isConversationControlled,searchTermIndex,isQuote,text_clean,text_normalized,tokens,tokens_no_stopwords,tokens_lemmatized,label,skor
str,f64,str,str,str,str,i64,i64,i64,i64,i64,str,str,i64,bool,f64,f64,f64,str,bool,str,str,str,str,str,str,str,str,str,bool,i64,bool,str,str,str,str,str,str,f64
"""tweet""",2.1153e18,"""https://x.com/arifin34533/stat…","""https://twitter.com/arifin3453…","""MATI‼️ https://t.co/Tn9mf4vwE8""",null,0,0,0,0,4,"""Wed Oct 28 04:56:49 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""","""{'binding_values': [{'key': 'p…","""{}""","""{'urls': [{'display_url': 'tem…",null,null,null,null,false,0,null,"""MATI‼""","""mati!!""","""mati""","""mati""","""mati""","""negatif""",0.7962
"""tweet""",2.1153e18,"""https://x.com/trynthi_/status/…","""https://twitter.com/trynthi_/s…","""Lengkap banget perlengkapannya…",null,0,0,0,0,15,"""Wed Oct 28 04:56:06 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,null,null,null,false,0,null,"""Lengkap banget perlengkapannya…","""lengkap sekali perlengkapannya…","""lengkap sekali perlengkapannya…","""lengkap perlengkapannya ngadep…","""lengkap perlengkapan ngadepin …","""negatif""",0.9402
"""tweet""",2.1153e18,"""https://x.com/tempodotco/statu…","""https://twitter.com/tempodotco…","""Baca: https://t.co/TVLqbdOxCs""",null,0,0,1,0,60,"""Wed Oct 28 04:56:05 +0000 2026""","""in""",0,true,2.1153e18,2.1153e18,1.8129942e7,"""tempodotco""",false,"""{'type': 'user', 'userName': '…","""{}""","""{'binding_values': [{'key': 'p…","""{}""","""{'urls': [{'display_url': 'tem…","""{'rest_id': '18129942', 'resul…",null,null,null,false,0,null,"""Baca:""","""baca:""","""baca""","""baca""","""baca""","""positif""",0.6085
"""tweet""",2.1153e18,"""https://x.com/twit_boongan/sta…","""https://twitter.com/twit_boong…","""@PocongGaib @prabowo @DivHumas…",null,0,0,0,0,4,"""Wed Oct 28 04:55:21 +0000 2026""","""in""",0,true,2.1153e18,2.1153e18,8.7500011e7,"""PocongGaib""",false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{'user_mentions': [{'id_str': …","""{'rest_id': '87500011', 'resul…",null,null,null,false,0,null,"""dari segitu banyaknya tni dan …","""dari segitu banyaknya tni dan …","""dari segitu banyaknya tni dan …","""segitu banyaknya tni polisi kn…","""segitu banyak tni polisi knp t…","""negatif""",0.9978
"""tweet""",2.1153e18,"""https://x.com/candysvga/status…","""https://twitter.com/candysvga/…","""14ribu orang ini kalau diturun…",null,0,0,0,0,4,"""Wed Oct 28 04:54:45 +0000 2026""","""in""",0,false,null,2.1153e18,null,null,false,"""{'type': 'user', 'userName': '…","""{}""",null,"""{}""","""{}""",null,"""{'rest_id': '21152711581532737…","""{'type': 'tweet', 'id': '21152…",null,false,0,true,"""14ribu orang ini kalau diturun…","""14ribu orang ini kalau diturun…","""14ribu orang ini kalau diturun…","""14ribu orang diturunkan padami…","""14ribu orang turun padamin bak…","""negatif""",0.9967
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tweet""",2.1146e18,"""https://x.com/officialinews_/s…","""https://twitter.com/officialin…","""Kebakaran hutan dan lahan (kar…",null,2,1,9,0,844,"""Mon Oct 26 02:56:06 +0000 2026""","""in""",0,false,null,2.1146e18,null,null,false,"""{'type': 'user', 'userName': '…","""{'media': [{'additional_media_…",null,"""{}""","""{'hashtags': [{'indices': [728…",null,null,null,null,false,0,null,"""Kebakaran hutan dan lahan (kar…","""kebakaran hutan dan lahan (kar…","""kebakaran hutan dan lahan karh…","""kebakaran hutan lahan karhutla…","""bakar hutan lahan karhutla kaw…","""positif""",0.633

In [ ]:
# Memeriksa struktur dan tipe data
print(df.schema)
# Memeriksa jumlah data
print("Jumlah data:", df.height)
# Memeriksa data kosong
print(df.null_count())
# Memeriksa distribusi label
print(df.group_by("label").len().sort("label"))



### Menampilkan Satu Kolom Saja
Kita bisa memilih satu kolom tertentu (misalnya kolom `tokens_lemmatized` atau kolom `label`) menggunakan fungsi `.select()` dari Polars.

In [20]:
# Contoh menampilkan kolom 'tokens_lemmatized' saja
display(df.select("tokens_lemmatized"))

tokens_lemmatized
str
"""mati"""
"""lengkap perlengkapan ngadepin …"
"""baca"""
"""segitu banyak tni polisi knp t…"
"""14ribu orang turun padamin bak…"
…
"""bakar hutan lahan karhutla kaw…"
"""halo sobat bmkg 25 oktober 202…"
"""kesan tidak jaga paru-paru bak…"


In [29]:
# Memperbaiki nama kolom dari 'sentiment' menjadi 'label'
display(df.select("tokens_lemmatized", "label"))

tokens_lemmatized,label
str,str
"""mati""","""negatif"""
"""lengkap perlengkapan ngadepin …","""negatif"""
"""baca""","""positif"""
"""segitu banyak tni polisi knp t…","""negatif"""
"""14ribu orang turun padamin bak…","""negatif"""
…,…
"""bakar hutan lahan karhutla kaw…","""positif"""
"""halo sobat bmkg 25 oktober 202…","""positif"""
"""kesan tidak jaga paru-paru bak…","""negatif"""


In [32]:
# Menggunakan Polars (pl) karena variabel df kita adalah Polars DataFrame
# Kita akan memetakan label string (misalnya 'positif' -> 'Positif', 'negatif' -> 'Negatif')
# atau memformat nama kolom hasil prediksi secara langsung.

df_labeled = df.with_columns(
    pl.when(pl.col("label") == "positif")
    .then(pl.lit("Positif"))
    .otherwise(pl.lit("Negatif"))
    .alias("sentiment_label")
)

# Menampilkan tokens beserta label sentimennya
display(df_labeled.select("tokens_lemmatized", "sentiment_label"))

tokens_lemmatized,sentiment_label
str,str
"""mati""","""Negatif"""
"""lengkap perlengkapan ngadepin …","""Negatif"""
"""baca""","""Positif"""
"""segitu banyak tni polisi knp t…","""Negatif"""
"""14ribu orang turun padamin bak…","""Negatif"""
…,…
"""bakar hutan lahan karhutla kaw…","""Positif"""
"""halo sobat bmkg 25 oktober 202…","""Positif"""
"""kesan tidak jaga paru-paru bak…","""Negatif"""


In [33]:
# Menentukan path file output CSV
OUTPUT_PATH = "/content/hasil_analisis_sentimen.csv"

# Menyimpan DataFrame df_labeled ke CSV menggunakan Polars
df_labeled.write_csv(OUTPUT_PATH)
print(f"Data berhasil disimpan ke {OUTPUT_PATH}")

# Mengunduh file langsung ke komputer (opsional, khusus di Google Colab)
from google.colab import files
files.download(OUTPUT_PATH)

Data berhasil disimpan ke /content/hasil_analisis_sentimen.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from sklearn.model_selection import train_test_split

# Memisahkan fitur dan label menggunakan kolom teks yang ada di data (text_normalized)
X = df["text_normalized"].to_list()
y = df["label"].to_list()

# Membagi dataset menjadi training (80%) dan testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
ngram_range=(1, 2),
min_df=1,
max_features=20000
)
# Membentuk representasi numerik data latih
X_train_tfidf = tfidf.fit_transform(X_train)
# Mengubah data uji menggunakan vocabulary data latih
X_test_tfidf = tfidf.transform(X_test)

In [26]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Karena kita sudah memprediksi seluruh baris menggunakan model BERT di awal,
# kita bisa langsung mengevaluasi subset data uji (y_test) dengan prediksi yang sesuai.

# Ambil label hasil prediksi BERT yang sudah kita jalankan sebelumnya
y_pred_all = df["label"].to_list()

# Dapatkan index untuk data uji (X_test) untuk mengambil subset prediksi yang sesuai
# Kita cari posisi teks X_test di dalam teks asli X
X_all = df["text_normalized"].to_list()
test_indices = [X_all.index(text) for text in X_test]
y_pred = [y_pred_all[idx] for idx in test_indices]

# Menghitung akurasi
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Menampilkan precision, recall, dan F1-score
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

# Menampilkan confusion matrix
print(confusion_matrix(
    y_test,
    y_pred,
    labels=list(set(y_test))
))

Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00       121
     positif       1.00      1.00      1.00        95

    accuracy                           1.00       216
   macro avg       1.00      1.00      1.00       216
weighted avg       1.00      1.00      1.00       216

[[ 95   0]
 [  0 121]]


In [27]:
teks = input("Masukkan review: ")
# Mengubah teks menjadi representasi TF-IDF
teks_tfidf = tfidf.transform([teks])
# Melakukan prediksi sentimen
hasil_prediksi = model.predict(teks_tfidf)
# Menampilkan hasil prediksi
print("\nHasil Analisis Sentimen")
print("-" * 40)
print("Review :", teks)
print("Sentimen :", hasil_prediksi[0])
print("-" * 40)


KeyboardInterrupt: Interrupted by user